##

> Week 1: From text to input embeddings

###

> 1. Data

In [10]:
import tiktoken
import kagglehub
from dotenv import load_dotenv
from pathlib import Path
import torch

from src.tokenizer import create_dataloader

torch.manual_seed(1234)

In [3]:
load_dotenv()
DATA_PATH = Path("../data")
DATA_PATH.mkdir(parents=True, exist_ok=True)
DATA_NAME = "kaushaltiwari/tiny-shakespeare"

# Download Data

path = kagglehub.dataset_download(
    handle=DATA_NAME,
    output_dir=DATA_PATH.as_posix(),
)

In [4]:
# Tokenizer using gt2
full_path = DATA_PATH / "tiny-shakespeare.txt"
with open(full_path, "r", encoding="utf-8") as f:
    text = f.read()

###

> 2. Tokens

A toy word-level tokenizer: a dictionary that maps each word to an id, with `<unk>` for unknown words.

In [ ]:
# To make it simple I will create a Toy Dictioary 
toy_text= "I love my kids with all my hear"
splitted_text = toy_text.split(" ")
tokenizer = {word: idx for idx, word in enumerate(set(splitted_text))}
tokenizer["<unk>"] = len(tokenizer)
print(f"My tokenizer size `{len(tokenizer)}` {tokenizer}")

My tokenizer size `8` {'my': 0, 'hear': 1, 'I': 2, 'love': 3, 'all': 4, 'with': 5, 'kids': 6, '<unk>': 7}


###

> 3. Manual implementation

Building (mimicking) a vector embedding by hand.

####

> What is an embedding?

An embedding maps each token id to a dense vector of real numbers.
It is stored as a matrix W of shape (N, M), where row i is the vector of token i:

- N = vocabulary size of the tokenizer (50257 for GPT-2 BPE), regardless of
how many of those tokens actually appear in our text.

- M = embedding dimension (e.g. 768 in GPT-2 small). It is a hyperparameter
we choose before training.

W starts with random values and is learned together with the rest of the model.
If we trained our own tokenizer on Tiny Shakespeare, N would be that tokenizer's
vocabulary size. A word-level vocabulary is impractical: any unseen word would
be out of vocabulary, which is why subword tokenizers like BPE are used.


In [37]:
# Embdding Matrix 
dim = 3
ids = torch.tensor([tokenizer[w] for w in splitted_text])
W = torch.arange(len(tokenizer) * dim, dtype=torch.float32).reshape(len(tokenizer), 3)
W

tensor([[ 0.,  1.,  2.],
        [ 3.,  4.,  5.],
        [ 6.,  7.,  8.],
        [ 9., 10., 11.],
        [12., 13., 14.],
        [15., 16., 17.],
        [18., 19., 20.],
        [21., 22., 23.]])

In [ ]:
# Create One hot encoding 
onehot = []
for idx in ids.tolist():
    row = [0 * len(tokenizer)] * len(tokenizer)
    row[idx] = 1
    onehot.append(row)


[[0, 0, 1, 0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0, 1, 0, 0],
 [0, 0, 0, 0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0, 0, 0, 0]]

In [59]:
# Now create the embedding layer using the embedding matrix W
torch.tensor(onehot).float() @ W

tensor([[ 6.,  7.,  8.],
        [ 9., 10., 11.],
        [ 0.,  1.,  2.],
        [18., 19., 20.],
        [15., 16., 17.],
        [12., 13., 14.],
        [ 0.,  1.,  2.],
        [ 3.,  4.,  5.]])

###

> 4. Linear implementation

The same lookup, now as a `Linear` layer applied to one-hot vectors.

In [65]:
# Transform the ids into one-hot encoding using PyTorch's built-in function
onehot = torch.nn.functional.one_hot(ids, num_classes=len(tokenizer)).float()

# Now create the embedding layer using the embedding matrix W
linear = torch.nn.Linear(len(tokenizer), dim, bias=False)
linear.weight = torch.nn.Parameter(W.T)
linear(onehot)

tensor([[ 6.,  7.,  8.],
        [ 9., 10., 11.],
        [ 0.,  1.,  2.],
        [18., 19., 20.],
        [15., 16., 17.],
        [12., 13., 14.],
        [ 0.,  1.,  2.],
        [ 3.,  4.,  5.]], grad_fn=<MmBackward0>)

###

> 5. Implementation with torch

Now using `torch.nn.Embedding`.

###

> 6. Data loader

This creates the dataset for training and testing. The challenge here is choosing `batch_size`, `max_length` and `stride`.

In [5]:
print(f"This is the dataset we are using: {DATA_NAME}")

# MaxLength is the maximum length of the sequence we want to feed into the model.
max_length = 128

# Generate Data
dataloader = create_dataloader(
    data=text,
    batch_size=64,
    max_length=max_length,
    stride=max_length, # This avoid Overlapping Sequences
)

data_iter = iter(dataloader)
inputs, target = next(data_iter)
print("Inputs:\n", inputs)
print("target:\n", target)

This is the dataset we are using: kaushaltiwari/tiny-shakespeare
Inputs:
 tensor([[  475,   284,   262,  ...,   198,    47,  2662],
        [ 1528,  3750,   198,  ...,   286,   340,  2637],
        [ 2514,  1037,   502,  ...,    11,   617,   286],
        ...,
        [  407,  4167,    13,  ...,   502,   616, 14412],
        [  835,   422, 27012,  ...,   275,  1000,    13],
        [ 1049,  1822,   418,  ...,    51,  1797,  5603]])
target:
 tensor([[  284,   262,   966,  ...,    47,  2662, 11401],
        [ 3750,   198,  6385,  ...,   340,  2637,   198],
        [ 1037,   502,   706,  ...,   617,   286,   345],
        ...,
        [ 4167,    13,  1867,  ...,   616, 14412,    11],
        [  422, 27012,    13,  ...,  1000,    13,   198],
        [ 1822,   418,   444,  ...,  1797,  5603,    25]])


###

> 7. Embedding layer and positional embedding

####

> Token embedding

In [6]:
# Embeddings
output_dim = 256
vocab_size = tiktoken.get_encoding("gpt2").n_vocab  # 50257

# Embedding matrix W of shape (vocab_size, output_dim), randomly initialized.
# It is learned during training together with the rest of the model.
token_embedding_layer = torch.nn.Embedding(num_embeddings=vocab_size, embedding_dim=output_dim)

# Lookup: each token id in `inputs` selects its row of W
token_embedding = token_embedding_layer(inputs)
batch_size, seq_len, emb_dim = token_embedding.shape
print(f"Embedding output [batch_size, seq_len, emb_dim]: {token_embedding.shape}")
print(f"{batch_size} sequences, {seq_len} tokens each, {emb_dim} numbers per token")


Embedding output [batch_size, seq_len, emb_dim]: torch.Size([64, 128, 256])
64 sequences, 128 tokens each, 256 numbers per token


####

> Positional embedding

In [8]:
# Generate Positional Econding 
post_embedding_layer = torch.nn.Embedding(num_embeddings=max_length, embedding_dim=output_dim)
post_embedding_layer

Embedding(128, 256)

####

> Input embedding (token + position)

From this point adding the positional encoding into the vector embedding to shift | move the vector for the transformer to identified the best vector position
we just sum them into the vector space , this would affect all the batches 

In [9]:
input_embedding = token_embedding + post_embedding_layer(torch.arange(seq_len))
input_embedding.shape

torch.Size([64, 128, 256])